In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms

from collections import defaultdict

from train_bpe import *

c:\Users\weiwe\anaconda3\envs\bpe\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_size = 1

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [3]:
def reshape_to_tuples(data, dim):
    if isinstance(data, torch.Tensor):
        if data.shape[0] == 1:
            data = data.squeeze().numpy()
    # data = (data * 255).astype(int)
    rows, cols = data.shape
    row_group_size = rows // dim[0]
    col_group_size = cols // dim[1]
    
    row_indices = []
    col_indices = []
    for i in range(0, row_group_size):
        row_indices.append([j for j in range(i, rows, row_group_size)])
        
    for i in range(0, col_group_size):
        col_indices.append([j for j in range(i, cols, col_group_size)])

    # print(row_indices)
    # print(col_indices)

    tuples = []
    indices_matrix = []

    for row_indices_group in row_indices:
        for col_indices_group in col_indices:
            group = []
            indices_tuple = []

            for r_idx in row_indices_group:
                for c_idx in col_indices_group:
                    group.append(data[r_idx, c_idx])
                    indices_tuple.append((r_idx, c_idx))
            
            indices_matrix.append(tuple(indices_tuple))
            tuples.append(tuple(group))
            
    return tuples, indices_matrix

In [4]:
def freq_pair(tuple_list):
    pairs = defaultdict(int)
    for tokens in tuple_list:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i+1])
            pairs[pair] += 1
    return pairs

In [5]:
def max_freq_pair(freq_pairs_dic):
    max_freq = None

    for pair, freq in freq_pairs_dic.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    return best_pair, max_freq

In [6]:
def merge(tuple_list, indices_matrix, pair, idx):
    new_tuple_list = []
    new_indices_matrix = []
    for i in range(len(tuple_list)):
        tokens_tuple = tuple_list[i]
        token_indices_tuple = indices_matrix[i]
        merged_tokens = []
        merged_indices = []
        
        j = 0
        while j < len(tokens_tuple):
            if j < len(tokens_tuple) - 1 and (tokens_tuple[j], tokens_tuple[j+1]) == pair:
                merged_tokens.append(idx)
                merged_indices.append(token_indices_tuple[j])
                j += 2
            else:
                merged_tokens.append(tokens_tuple[j])
                merged_indices.append(token_indices_tuple[j])
                j += 1

        new_tuple_list.append(tuple(merged_tokens))
        new_indices_matrix.append(tuple(merged_indices))
    return new_tuple_list, new_indices_matrix

In [7]:
def train(tokens_tuple_list, indices_matrix, vocab_size, min_freq=2):
    vocab = defaultdict(int)

    while len(vocab) < vocab_size:
        pair, freq = max_freq_pair(freq_pair(tokens_tuple_list))

        if freq < min_freq:
            break

        if pair not in vocab.values():
            idx = len(vocab) + 1
            vocab[idx] = pair
        else:
            for key, val in vocab.items():
                if val == pair:
                    idx = key
                    
        tokens_tuple_list, indices_matrix = merge(tokens_tuple_list, indices_matrix, pair, idx)
    
    return tokens_tuple_list, indices_matrix, vocab

In [19]:
dim = (4, 4)

for image, label in tqdm(train_loader):
    ttl, im = reshape_to_tuples(image, dim)
    new_ttl, new_im, vocab = train(ttl, im, 1000)
    break

  0%|          | 0/60000 [00:00<?, ?it/s]


In [20]:
vocab

defaultdict(int,
            {1: (0.0, 0.0),
             2: (1, 1),
             3: (2, 0.0),
             4: (2, 1),
             5: (1, 0.0),
             6: (2, 2),
             7: (0.99607843, 5),
             8: (0.99607843, 1),
             9: (2, 3),
             10: (2, 4),
             11: (4, 8),
             12: (0.65882355, 5),
             13: (0.9372549, 0.0),
             14: (3, 0.99607843),
             15: (6, 6),
             16: (0.99607843, 0.0),
             17: (7, 0.6117647),
             18: (17, 0.0),
             19: (1, 0.99607843),
             20: (0.3764706, 3),
             21: (0.16862746, 9),
             22: (4, 0.0),
             23: (3, 0.03529412),
             24: (5, 12),
             25: (3, 0.79607844),
             26: (25, 7),
             27: (0.99607843, 3),
             28: (14, 7),
             29: (3, 0.09019608)})